# day03

## 1. plotly
 : 인터렉티브 차트를 만들 수 있는 고급 시각화 파이썬 라이브러리
   줌, 팬, 호버 등 다양한 인터렉션을 지원

+) plotly 설치
	vs Code 터미널
	python -m pip install plotly

## 2. 상태 관리 및 성능 최적화

### 1) 세션 상태 관리(st.session_state)
 : 세션 상태를 사용하면 사용자의 입력이나 계산 결과를
   앱 전체에서 유지할 수 있다

### 2) 세션 상태 초기화
	if 'key' not in st.session_state : 조건으로 초기화
	앱이 처음 실행될 때만 초기값 설정

### 3) 세션 상태 사용
	st.session_state.key : 값 접근
	st.session_state['key']
	=> 딕셔너리 처럼 사용

### 4) st.rerun()
	페이지를 새로고침하여 업데이트
	세션 상태 변경 후 업데이트에 사용

### 5) 세션 상태의 장점
	- 버튼 클릭 후에도 값 유지
	- 여러 위젯 간 데이터 공유

## 3. 캐싱을 통한 성능 최적화
 : 반복적인 계산을 피하고 앱의 성능을 크게 향상시킬 수 있다

@st.cache_data vs @st.cache_resource
- @st.cache_data : 데이터를 캐시(DataFrame, 변수 등)
- @st.cache_resource : 리소스를 캐시(모델 연결 등)
- 캐싱의 장점
	반복 계산 방지
	앱 성능 향상

## 4. 프로그래스 바 및 스피너
 : 긴 작업을 수행할 때 사용자에게 진행 상황을 보여주는 기능

### 1) st.progress()
	- 프로그래스 바 생성
	- 0.0 ~ 1.0 사이의 값으로 표시
	- .progress(value)로 업데이트

### 2) st.spinner()
	- 스피너 표시(로딩 애니메이션)
	- with문과 사용
	- 작업이 완료되면 자동으로 사라짐

### 3) st.empty()
	- 빈 컨테이너 생성
	- 동적으로 내용 업데이트에 사용
	- .text(), write() 등으로 내용 변경

### 실습

In [ ]:
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

st.title("plotly 인터랙티브 차트")

# 데이터 생성
dates = pd.date_range('2024-01-01', periods=30, freq="D")
df = pd.DataFrame({
    "날짜" : dates,
    "매출" : np.random.randint(50, 200, 30),
    "비용" : np.random.randint(30, 120, 30),
    "카테고리" : np.random.choice(["A", "B", "C"], 30)
})

st.dataframe(df)

# 1. 꺽은 선 그래프
st.header("1. 선 그래프")
fig_line = px.line(
    df, 
    x='날짜',
    y='매출',
    title="매출추이"
)
st.plotly_chart(fig_line)

# 2. 막대 그래프
st.header("2. 막대 그래프")
fig_bar = px.bar(
    df.groupby('카테고리')['매출'].mean().reset_index(),
    x="카테고리",
    y='매출',
    title='카테고리별 평균 매출',
    color='카테고리',
    color_discrete_map={'A':'blue', "B":'green', 'C':'red'}
)
st.plotly_chart(fig_bar)

# 3. 산점도
st.header("3. 산점도")
fig_scatter = px.scatter(
    df,
    x='매출',
    y='비용',
    title='매출 vs 비용',
    size='매출',
    hover_data=['날짜']
)

st.plotly_chart(fig_scatter)

# 4. 여러 시리즈를 포함한 그래프(plotly graph objects)
st.header("4. 여러 시리즈 비교")
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['날짜'],
    y=df['매출'],
    mode='lines+markers',
    name='매출',
    line=dict(color="red", width=2)
))

fig.add_trace(go.Scatter(
    x = df['날짜'],
    y= df['비용'],
    mode='lines+markers',
    name='비용',
    line=dict(color="blue", width=2)
))

st.plotly_chart(fig)

In [ ]:
import streamlit as st

st.title("세션 상태 관리")

# 세션 상태 초기화
if 'counter' not in st.session_state:
    st.session_state.counter = 0
if 'name' not in st.session_state:
    st.session_state.name = ""

st.header("1. 카운터")
col1, col2, col3 = st.columns(3)

with col1 :
    if st.button("증가"):
        st.session_state.counter += 1
with col2:
    if st.button("감소"):
        st.session_state.counter -= 1
with col3:
    if st.button("초기화"):
        st.session_state.counter = 0

st.write(f"현재 카운터 : {st.session_state.counter}")

# 이름 저장
st.header("2. 이름 저장")
name = st.text_input("이름을 입력하세요",
                     value=st.session_state.name)
                    # 세션에 있는 name의 값을 기본값으로 설정
if name:
    st.session_state.name = name
    st.write(f"저장된 이름 : {st.session_state.name}")

# 리스트 관리
st.header("3. 리스트 관리")
if 'items_li' not in st.session_state:
    st.session_state.items_li = []

new_item = st.text_input("새 항목 추가")
if st.button("추가"):
    if new_item:
        st.session_state.items_li.append(new_item)
        st.success(f"{new_item} 추가됨!")

if st.session_state.items_li:
    st.write("항목 목록 : ")
    for i, item in enumerate(st.session_state.items_li):
        col1, col2, = st.columns([3, 1]) # 열 너비 설정
        with col1:
            st.write(f"{i+1}. {item}")
        with col2:
            if st.button("삭제", key=f"delete_{i}"):
                st.session_state.items_li.pop(i)
                st.rerun() # 페이지 새로고침 => 변경된 값 적용

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import time

st.title("캐싱을 통한 성능 최적화")

# 캐싱 없이 데이터 로드
st.header("1. 캐싱 없이 데이터 로드")

@st.cache_data # 이 데코레이터를 주석 처리 하면 느려짐
def load_data():
    # 데이터를 불러오는 함수
    time.sleep(2) # 2초 대기
    df = pd.DataFrame({
        "A":np.random.randn(1000),
        "B":np.random.randn(1000),
        "C":np.random.randn(1000)
    })
    return df

if st.button("데이터 로드(캐시 사용)"):
    start_time = time.time()
    df = load_data()
    elapsed_time = time.time() - start_time
    st.write(f"로드 시간 : {elapsed_time : .2f}초")
    st.dataframe(df.head())

# 캐싱된 데이터 사용
if 'df' in st.session_state:
    st.write("캐싱된 데이터 사용 (즉시 표시)")
    st.dataframe(st.session_state.df.head())

# 복잡한 계산 캐싱
st.header("2. 복잡한 계산 캐싱")
@st.cache_data
def expensive_calculation(n):
    time.sleep(1) # 1초 대기
    return sum(range(n))

n = st.number_input("숫자 입력", min_value=1, max_value=1000000, value=1000000)

if st.button("계산 실행"):
    start_time = time.time()
    result = expensive_calculation(int(n))
    elapsed_time = time.time() - start_time
    st.write(f"결과 : {result}")
    st.write(f"계산 시간 : {elapsed_time : .2f}")
    st.info("같은 숫자로 다시 계산하면 즉시 결과가 나온다(캐싱)")

# 모델 로드 캐싱
st.header("3. 모델 로드 캐싱")
@st.cache_resource # 리소스 캐싱
def load_model():
    time.sleep(2)
    # 실제로는 model = joblib.load("model.pk1")
    return "모델이 로드되었습니다"

if st.button("모델 로드"):
    model = load_model()
    st.success(model)
    st.info("모델은 한번만 로드되고 이후에는 캐시에서 가져온다")

In [ ]:
import streamlit as st
import time
import numpy as np

st.title("프로그래스 바 및 스피너")

# 프로그래스 바
st.header("1. 프로그래스 바")
if st.button("작업 시작(프로그래스 바)"):
    progress_bar = st.progress(0)
    status_text = st.empty()

    # 작업 시뮬레이션
    for i in range(100):
        time.sleep(0.01)
        progress_bar.progress(i+1) # 1%씩 증가
        status_text.text(f"진행률 : {i+1}%") 
    st.success("작업 완료!")

# 스피너
st.header("2. 스피너")
if st.button("작업 시작(스피너)"):
    with st.spinner("작업중... 잠시만 기다려주세요"):
        time.sleep(3)
        # 실제 작업 코드
        result = sum(range(1000000))
    st.success(f"작업 완료! 결과 : {result}")

# 프로그래스 바와 상태 텍스트 조합
st.header("3. 고급 프로그래스 표시")
if st.button("복잡한 작업 시작"):
    progress_bar = st.progress(0)
    status_text = st.empty()

    steps = ["데이터 로드", '전처리', '모델 학습', '예측', '결과 저장']
    step_idx = 0
    for i, step in enumerate(steps):
        status_text.text(f"진행 중 : {step}...")
        time.sleep(1)
        progress_bar.progress((i+1) / len(steps))

    status_text.text("완료!")
    st.success("모든 작업이 완료 되었습니다.")

## 5. 개인 프로젝트 주제 선정

### 프로젝트 제목
**폭염 시 고령층 온열질환 위험도 예측 시스템**

### 프로젝트 개요
기상청의 과거 기상 데이터와 질병관리청의 온열질환 발생 데이터를 활용하여 폭염 시 고령층의 온열질환 위험도를 예측하는 머신러닝 프로젝트입니다.

### 프로젝트 목표
- 기온, 습도, 풍속 등의 기상조건을 분석해 온열질환 위험도 예측
- Streamlit을 활용하여 대시보드 구현
- 고령자의 안전한 외출을 지원하는 서비스 제공

### 데이터 수집 일정
**마감: 9월 1일(화)**

⚠️ 얼마 안 남았으니 빠르게 정하고 내일 자료를 찾아야 한다!